# AAL Sales Analysis · Q4 2020

**Applied Data Science with Python | Course-end project**  
**Working style:** follow the assignment's four sections in order. Sections 1 and 2 are complete, and Section 3 is now ready to run.

| Rubric section | Status |
| --- | --- |
| **1. Data Wrangling** (a–d) | Complete |
| **2. Data Analysis** (a–d and extensions) | Complete |
| **3. Data Visualization** | Ready to run |
| **4. Report Generation** | To be completed |

The data describes apparel sales in Australia. Its source does **not** specify a currency, so we refer to the `Sales` column as recorded sales values without assuming AUD.

**Before running:** place the supplied `AusApparalSales4thQrt2020.csv` beside this notebook in Practice Labs. If working from the downloaded project folder, leave the original ZIP in the `data/` directory. Both arrangements are supported.

## 1. Data Wrangling

**Rubric goal:** (a) check missing/incorrect records, (b) justify how to handle them, (c) apply normalization, and (d) demonstrate and explain `GroupBy()`.

### 1a. Inspect missing and incorrect entries

**Step 1: Load the original dataset.** `pandas` (usually imported as `pd`) reads the CSV into a *DataFrame*: a table with rows and columns. This cell also accepts the original ZIP if you are working from the project folder.

In [ ]:
from pathlib import Path
import pandas as pd

# The first existing file wins. Practice Labs usually keeps CSV and notebook together.
candidates = [
    Path('AusApparalSales4thQrt2020.csv'),
    Path('data/AusApparalSales4thQrt2020.csv'),
    Path('../data/AusApparalSales4thQrt2020.csv'),
    Path('1739523823_ausapparalsales4thqrt2020.zip'),
    Path('data/1739523823_ausapparalsales4thqrt2020.zip'),
    Path('../data/1739523823_ausapparalsales4thqrt2020.zip'),
]
source_file = next((path for path in candidates if path.is_file()), None)
if source_file is None:
    raise FileNotFoundError('Put the CSV beside this notebook, or the original ZIP in data/.')

raw_df = pd.read_csv(source_file)
print('Source file:', source_file)
print(f'Rows: {len(raw_df):,}; columns: {len(raw_df.columns)}')
display(raw_df.head())

**Step 2: Check data types and missing cells.** `isna()` gives `True` for missing values; `notna()` does the opposite. Summing Boolean values counts them.

In [ ]:
print('Original data types:')
display(raw_df.dtypes.to_frame('dtype'))

missing_check = pd.DataFrame({
    'missing: isna()': raw_df.isna().sum(),
    'present: notna()': raw_df.notna().sum(),
})
display(missing_check)

**Step 3: Clean a copy, not the source.** The text fields have leading spaces, so we remove them with `.str.strip()`. We convert dates and numeric fields explicitly. `errors='coerce'` turns an invalid entry into a detectable missing value rather than hiding a parsing error.

In [ ]:
df = raw_df.copy()  # The original raw_df stays unchanged.
df.columns = df.columns.str.strip()

for column in ['Time', 'State', 'Group']:
    df[column] = df[column].astype('string').str.strip()

df['Date'] = pd.to_datetime(
    df['Date'].astype('string').str.strip(), format='%d-%b-%Y', errors='coerce'
)
for column in ['Unit', 'Sales']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Example of removing extra spaces:')
print('Before:', repr(raw_df.loc[0, 'Group']))
print('After: ', repr(df.loc[0, 'Group']))

display(df.head())

**Step 4: Inspect potential errors after cleaning.** A *duplicate business key* would mean more than one entry for the same Date × Time × State × Group. Negative sales might be returns in another business, so we flag them for investigation rather than deleting them automatically.

In [ ]:
print('Missing or invalid values after conversion:')
display(df.isna().sum().to_frame('Count'))

check_keys = ['Date', 'Time', 'State', 'Group']
quarter_start = pd.Timestamp('2020-10-01')
quarter_end = pd.Timestamp('2020-12-31')

issues = pd.Series({
    'Identical duplicate rows': int(df.duplicated().sum()),
    'Duplicate date/time/state/group keys': int(df.duplicated(check_keys).sum()),
    'Out-of-quarter dates': int((df['Date'].notna() & ~df['Date'].between(quarter_start, quarter_end)).sum()),
    'Units <= 0': int((df['Unit'] <= 0).sum()),
    'Fractional units': int(((df['Unit'] % 1 != 0) & df['Unit'].notna()).sum()),
    'Negative sales values': int((df['Sales'] < 0).sum()),
    'Unexpected group labels': int((df['Group'].notna() & ~df['Group'].isin(['Kids', 'Men', 'Women', 'Seniors'])).sum()),
    'Unexpected time labels': int((df['Time'].notna() & ~df['Time'].isin(['Morning', 'Afternoon', 'Evening'])).sum()),
})
display(issues.to_frame('Rows flagged'))

for column in ['State', 'Group', 'Time']:
    print(column, ':', sorted(df[column].dropna().unique().tolist()))

**Step 5: Check whether any *whole days* are absent.** A dataset can have zero `NaN` cells while missing entire days of trading. Q4 2020 has **92 calendar days**, so we compare that calendar with the distinct dates in the file.

In [ ]:
quarter_dates = pd.date_range('2020-10-01', '2020-12-31', freq='D')
observed_dates = pd.DatetimeIndex(df['Date'].dropna().unique())
missing_dates = quarter_dates.difference(observed_dates)

print('Recorded calendar days:', len(observed_dates), '/', len(quarter_dates))
print('Unrecorded dates:')
for day in missing_dates:
    print(' ', day.strftime('%d %b %Y'))

rows_each_day = df.groupby('Date').size()
print('Rows per recorded day: minimum', rows_each_day.min(), 'maximum', rows_each_day.max())

### 1b. Recommend how to treat missing or incorrect information

**Decision for this dataset:**

| Finding | Recommended treatment | Reason |
| --- | --- | --- |
| Missing values in existing rows | **No filling or dropping required** if the checks above show zero | We should not change valid data without a reason. |
| Extra spaces in text fields | **Strip whitespace** | Otherwise `' Men'` and `'Men'` could be treated as different categories. |
| Incorrect dates, amounts or duplicate keys (if found later) | **Investigate against the source before dropping or filling** | Automatic fixes can create false business results. |
| Entire dates absent (31 October and 31 December 2020) | **Keep them identified as unrecorded; do not fill with zero or invented averages** | A missing day does not prove the business earned no sales. |

**Hypothesis, unverified:** 31 October is Halloween and 31 December is New Year's Eve. Holiday-related closures or altered trading *could* explain the two gaps, but a reporting or extraction omission is also possible. The file contains no opening-hours data. Halloween is not generally a day when Australian retailers close, so neither date should be labelled a confirmed closure without evidence.

**Why not `dropna()` or `fillna()` here?** Those functions operate on *existing rows*, whereas two entire days are missing from the file. We would need independent trading schedules or original transaction records to resolve that question.

### 1c. Apply Min-Max normalization

**Chosen method: Min-Max normalization**, as recommended by the assignment. It rescales each numeric column to a value between **0 and 1**:

\[
 x_{\mathrm{normalized}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
\]

We will normalize **`Unit`** and **`Sales`**, leaving the originals intact. We should **not** normalize dates or category labels.

In [ ]:
for column in ['Unit', 'Sales']:
    minimum = df[column].min()
    maximum = df[column].max()

    if maximum == minimum:
        raise ValueError(f'Cannot apply Min-Max to {column}: all values are identical.')

    df[f'{column}_normalized'] = (df[column] - minimum) / (maximum - minimum)

print('Original and normalized values:')
display(df[['Unit', 'Unit_normalized', 'Sales', 'Sales_normalized']].head(8))

print('Normalized minimum and maximum (both must be 0 and 1):')
display(df[['Unit_normalized', 'Sales_normalized']].agg(['min', 'max']))

**Why might the two normalized columns be identical?** Inspect the ratio `Sales / Unit`. In this particular dataset, each sale value equals **2,500 × units**, so both columns carry the same relative information. Normalization does **not** make them independent.

In [ ]:
ratio_values = sorted((df['Sales'] / df['Unit']).dropna().unique().tolist())
print('Distinct Sales / Unit ratios:', ratio_values)
print('Greatest difference between normalized columns:',
      (df['Sales_normalized'] - df['Unit_normalized']).abs().max())

### 1d. Apply `GroupBy()` and recommend its use

`groupby()` uses **split → apply → combine**: split rows into logical groups, calculate a value per group, then combine the results into one summary table.

For AAL, **grouping (data chunking)** is more useful than merging at this stage. We already have all columns in one table, and we need state-wise and demographic summaries. A **merge** would be appropriate later if we obtained a *different table*, such as store locations or marketing spend.

**Important:** aggregate the original `Sales` and `Unit` values, not the normalized columns. Sums of normalized scores are not meaningful revenue totals.

In [ ]:
by_state = (
    df.groupby('State', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
)

by_state_and_group = (
    df.groupby(['State', 'Group'], as_index=False)
      .agg(Records=('Sales', 'size'), Total_Sales=('Sales', 'sum'))
      .sort_values(['State', 'Group'])
)

print('State-level groupings:')
display(by_state)
print('First ten state-by-group combinations:')
display(by_state_and_group.head(10))

**Section 1 verification.** The following quick checks are meant to catch mistakes before we begin the statistical analysis. They do not change any data.

In [ ]:
assert len(raw_df) == len(df), 'Rows must not disappear during cleaning.'
assert df[['Unit_normalized', 'Sales_normalized']].notna().all().all()
assert df['Unit_normalized'].between(0, 1).all()
assert df['Sales_normalized'].between(0, 1).all()
assert abs(by_state['Total_Sales'].sum() - df['Sales'].sum()) < 0.000001
assert (df['Unit_normalized'] - df['Sales_normalized']).abs().max() < 0.000000001
print('Section 1 checks passed: row count preserved, normalized values in [0,1], grouped sales reconciled.')

**Section 1 recap.** We checked data quality, documented the missing-date treatment, normalized without changing source values, and practised `GroupBy()`. Now we can analyse the *original* Sales and Unit columns.

---
## 2. Data Analysis

**Rubric goal:** (a) descriptive statistics for Unit and Sales, (b) identify highest/lowest selling demographic groups, (c) compare the highest/lowest selling states, and (d) prepare weekly, monthly and quarterly reports.

The original assignment repeats the group question in 2b and 2c. We use 2c to answer the wider project question about states, without ignoring the duplicate wording.

### 2a. Calculate descriptive statistics

We will find **mean**, **median**, **mode** and **standard deviation** for both numeric columns:

- **Mean:** arithmetic average (sum divided by number of rows).
- **Median:** the middle value when data is ordered.
- **Mode:** the value appearing most often.
- **Standard deviation:** how much values vary around the mean. Pandas `std()` uses the *sample* standard deviation (`ddof=1`).

Run this cell **after Section 1**, because it uses our cleaned `df`.

In [ ]:
# Calculate the summary for each original numeric column.
statistics = df[['Unit', 'Sales']].agg(['mean', 'median', 'std']).T
statistics['mode'] = [df[column].mode().iloc[0] for column in ['Unit', 'Sales']]
statistics = statistics[['mean', 'median', 'mode', 'std']]

print('Descriptive statistics for each recorded row:')
display(statistics.round(2))
print(f"Sales mode: {statistics.loc['Sales', 'mode']:,.0f}")

**What did we learn?** The recorded mean sale is about **45,013.56**, the median is **35,000**, and the most common sale value is **22,500** (currency unspecified). The mean being above the median suggests larger transactions pull the average upward. These are values **per dataset row**, not per individual shop or customer; the dataset does not identify transactions.

### 2b. Find the highest and lowest selling demographic groups

We will group the original **Sales** and **Unit** fields by `Group`. We use the *total* sales value to rank Kids, Men, Women and Seniors. Counts are included so we can see whether groups have similar numbers of records.

In [ ]:
group_report = (
    df.groupby('Group', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
      .reset_index(drop=True)
)

display(group_report)
print('Highest selling group:', group_report.iloc[0]['Group'])
print('Lowest selling group: ', group_report.iloc[-1]['Group'])

**Observation:** Men have the highest recorded total (**85,750,000**), and Seniors the lowest (**84,037,500**). Each demographic group contributes **1,890** records. The gap is modest compared with the overall size of the dataset; these totals alone do not explain *why* a group leads.

### 2c. Find the highest and lowest selling states or territories

The assignment's 2b and 2c repeat the group question, but the project statement explicitly asks for **state-by-state sales**. We address this here, ranking states by the sum of original Sales.

In [ ]:
state_report = (
    df.groupby('State', as_index=False)
      .agg(Records=('Sales', 'size'), Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
      .sort_values('Total_Sales', ascending=False)
      .reset_index(drop=True)
)

display(state_report)
print('Highest selling state:', state_report.iloc[0]['State'])
print('Lowest selling state: ', state_report.iloc[-1]['State'])

**Observation:** Victoria (VIC) is highest at **105,565,000** and Western Australia (WA) is lowest at **22,152,500**. Every recorded state/territory has **1,080** rows. We cannot infer store productivity or market demand without store counts, populations, or other context. The source does not include ACT.

**Investigate the lowest-selling states by demographic (part of 2c).**

Before making a marketing recommendation, we should ask: **within each lower-selling state, which customer group contributes the most and least?** We use `groupby(['State', 'Group'])` to build a state-by-demographic sales table.

These are *recorded sales totals*, not profit. Each state has the same number of rows for each demographic. We lack store counts or local market size.

In [ ]:
# Compare all four demographic groups within each state.
segment_sales = df.groupby(['State', 'Group'])['Sales'].sum().unstack('Group')
segment_share = 100 * segment_sales.div(segment_sales.sum(axis=1), axis=0)

# Choose the four states or territories with the lowest TOTAL recorded sales.
low_state_names = state_report.nsmallest(4, 'Total_Sales')['State']

low_state_demographics = pd.DataFrame({
    'Total_Sales': segment_sales.sum(axis=1),
    'Strongest_Group': segment_sales.idxmax(axis=1),
    'Strongest_Sales': segment_sales.max(axis=1),
    'Lowest_Group': segment_sales.idxmin(axis=1),
    'Lowest_Sales': segment_sales.min(axis=1),
    'Lowest_Share_pct': segment_share.min(axis=1).round(2),
    'Group_Gap_pp': (segment_share.max(axis=1) - segment_share.min(axis=1)).round(2),
}).loc[low_state_names].reset_index()

print('Sales by demographic in the four lowest-selling states:')
display(segment_sales.loc[low_state_names])
print('Summary: strongest, lowest, and the lowest group share in each state:')
display(low_state_demographics)
print('All-state within-state group share range:',
      f'{segment_share.min().min():.2f}% to {segment_share.max().max():.2f}%')

**What the breakdown suggests (hypotheses, not conclusions):**

| Lower-selling area | Strongest group | Lowest group | Sensible *test*, not a cut |
|---|---|---|---|
| WA | Men | Women | Test whether a Women's offer, assortment change or marketing message improves sales |
| NT | Men | Seniors | Investigate Seniors' needs and test a targeted offer |
| TAS | Kids | Women | Test Women's assortment or a family-linked campaign |
| QLD | Kids | Seniors | Investigate Seniors' demand while testing family-oriented promotions |

The key finding is **relative balance**: all four groups contribute approximately one quarter of each state's recorded sales. The weakest observed share is **23.76% (Women in WA)**; even the largest within-state difference (WA) is only **2.21 percentage points**. The data does **not** support removing any demographic group.

Revenue is not profit. Any decision to reduce a range needs gross margins, inventory costs, returns, store counts, local demand and footfall. Trial small measured interventions before reallocating entire demographic ranges.

### 2d. Produce weekly, monthly and quarterly reports

A **time-based GroupBy** creates reports at different levels of detail. We first convert each date to a weekly or monthly **period**; this labels which calendar bucket each row belongs to. We sum original sales and units and count distinct **recorded** days. No missing days are filled or estimated.

**Weekly:** We define weeks as **Monday to Sunday**, using `W-SUN`. The first and last weeks extend beyond Q4, so their totals only include Q4 records from the source; those rows are not comparable with complete weeks without adjusting for day counts.

In [ ]:
weekly_report = (
    df.assign(Week=df['Date'].dt.to_period('W-SUN'))
      .groupby('Week', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Week')
)

print('Weekly sales report (Monday-Sunday calendar weeks):')
display(weekly_report)
print('Number of weekly reporting periods:', len(weekly_report))

**Monthly:** Converting to a monthly period with `M` produces one total for **October, November and December**. `Recorded_Days` shows why two months have incomplete calendar coverage: October and December each have only 30 recorded days, and November has 30 too because it has 30 calendar days.

In [ ]:
monthly_report = (
    df.assign(Month=df['Date'].dt.to_period('M'))
      .groupby('Month', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Month')
)

print('Monthly sales report:')
display(monthly_report)

**Quarterly:** The `Q` period groups the entire quarter together. The quarter contains **90 recorded dates**, although October through December have **92 calendar dates**. We report recorded values only and flag the two missing dates as unverified gaps.

In [ ]:
quarterly_report = (
    df.assign(Quarter=df['Date'].dt.to_period('Q'))
      .groupby('Quarter', as_index=False)
      .agg(Recorded_Days=('Date', 'nunique'), Total_Units=('Unit', 'sum'),
           Total_Sales=('Sales', 'sum'))
      .sort_values('Quarter')
)

print('Quarterly sales report:')
display(quarterly_report)
print('Quarter total recorded sales:', f"{quarterly_report['Total_Sales'].sum():,.0f}")
print('Quarter labels:', quarterly_report['Quarter'].astype(str).tolist())

### 2e. Extra business question: Should we focus on higher or lower spend?

**Important distinction: we can measure higher- or lower-*volume recorded periods*, not individual customer spending.** Each row describes a date, state, demographic and time of day, aggregating units and sales. It is **not an individual customer** or basket. The dataset contains no transaction count, customer ID, variation in item prices, cost or profit, so we cannot calculate average basket value or establish whether individual shoppers spend more per visit.

Our first comparison is **average units per recorded period** for each state and group. Every state/group combination contains the same 270 observed records, so the means are comparable within each state. Because `Sales = Unit × 2,500` throughout, sales per record tells the same volume story, not willingness to pay.

In [ ]:
# An observation is a state/group/time/day aggregate, NOT a person or transaction.
state_group_intensity = (
    df.groupby(['State', 'Group'], as_index=False)
      .agg(Records=('Unit', 'size'),
           Average_Units_per_Record=('Unit', 'mean'),
           Average_Sales_per_Record=('Sales', 'mean'),
           Total_Sales=('Sales', 'sum'))
)
print('Average units and sales per recorded period in the lower-selling states:')
display(state_group_intensity[
    state_group_intensity['State'].isin(low_state_names)
].sort_values(['State', 'Average_Units_per_Record'], ascending=[True, False]).round(2))

**Next: compare higher- and lower-volume recorded periods within each state.** We calculate each state's own 25th and 75th percentiles of `Unit`, instead of comparing all states with Victoria's larger numbers.

- **Low:** `Unit` at or below that state's 25th-percentile cutoff.
- **High:** `Unit` at or above that state's 75th-percentile cutoff.
- **Middle:** everything between the cutoffs.

Values tied at the cutoff count in their band, so these are **not exactly quarters of all records**. They are **not quartiles of customer spending**.

In [ ]:
# Calculate state-specific quartile thresholds and label each record.
state_volume_thresholds = (
    df.groupby('State')['Unit']
      .quantile([0.25, 0.75])
      .unstack()
      .rename(columns={0.25: 'Low_cutoff_units', 0.75: 'High_cutoff_units'})
)
records_with_bands = df.join(state_volume_thresholds, on='State').copy()
records_with_bands['Volume_Band'] = 'Middle'
records_with_bands.loc[
    records_with_bands['Unit'] <= records_with_bands['Low_cutoff_units'], 'Volume_Band'
] = 'Low'
records_with_bands.loc[
    records_with_bands['Unit'] >= records_with_bands['High_cutoff_units'], 'Volume_Band'
] = 'High'

record_volume_bands = (
    records_with_bands.groupby(['State', 'Volume_Band'], as_index=False)
      .agg(Records=('Unit', 'size'), Mean_Units=('Unit', 'mean'),
           Total_Units=('Unit', 'sum'), Total_Sales=('Sales', 'sum'))
)
record_volume_bands['State_Sales_Share_pct'] = (
    100 * record_volume_bands['Total_Sales'] /
    record_volume_bands.groupby('State')['Total_Sales'].transform('sum')
)
print('Unit cutoffs for the four lower-selling states:')
display(state_volume_thresholds.loc[low_state_names])
print('Low, middle and high recorded periods by their sales contribution:')
display(record_volume_bands[
    record_volume_bands['State'].isin(low_state_names)
].sort_values(['State', 'Volume_Band']).round(1))

**Interpretation for our recommendations:** In WA, records of **11+ units** fall in the high-volume band and account for **42.5%** of recorded sales; records of **5 or fewer units** fall in the low-volume band and account for **12.4%**. They represent aggregated selling periods, not customer baskets.

Protect stock availability in busier periods; try small controlled promotions or assortment improvements in quieter periods. Discounts can lower margin or displace purchases that would have happened anyway, so measure **incremental profit**, not just increased revenue. We should not drop entire Women, Men, Kids or Seniors ranges based on this dataset.

**Data needed for a real high-spend-versus-low-spend customer strategy:** transaction count, receipt/basket-level quantities, prices actually paid, customer or visit identifiers (handled appropriately), store count, product mix, returns, cost of goods, margin, inventory and footfall. Without them, we cannot distinguish more visits from larger baskets or higher spending per customer.

**What do the time summaries suggest?** November has the lowest recorded monthly sales (**90,682,500**), while December has the highest (**135,330,000**). A seasonal explanation is *possible*, but one Q4 dataset does not demonstrate a recurring trend. We should avoid treating Halloween or New Year's Eve as confirmed closures.

### Section 2 verification

These checks make sure the totals reconcile across the demographic, geographic and time-based reports. The tables must all add back to the original source's recorded sales and units; none should create or discard data.

In [ ]:
for report in [group_report, state_report, weekly_report, monthly_report, quarterly_report]:
    assert report['Total_Sales'].sum() == df['Sales'].sum()
    assert report['Total_Units'].sum() == df['Unit'].sum()

assert weekly_report['Recorded_Days'].sum() == df['Date'].nunique()
assert monthly_report['Recorded_Days'].sum() == df['Date'].nunique()
assert quarterly_report['Recorded_Days'].sum() == df['Date'].nunique()

assert segment_sales.to_numpy().sum() == df['Sales'].sum()
assert (segment_share.sum(axis=1) - 100).abs().max() < 1e-8
assert record_volume_bands['Total_Sales'].sum() == df['Sales'].sum()
assert record_volume_bands['Records'].sum() == len(df)
assert (record_volume_bands.groupby('State')['State_Sales_Share_pct'].sum() - 100).abs().max() < 1e-8
print('Section 2 checks passed: all group, state and period totals reconcile.')

---
## 3. Data Visualization

**Rubric goal:** compare states and demographic groups in both directions; find peak and off-peak times of day; show daily, weekly, monthly and quarterly charts; and explain the visualization package. We also assemble a four-panel static dashboard. All plots use original recorded sales, not Min-Max scores. The source does not specify currency. Missing dates are gaps, **not zero sales**.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
state_order = state_report['State'].tolist()
customer_groups = ['Kids', 'Women', 'Men', 'Seniors']
time_order = ['Morning', 'Afternoon', 'Evening']

### 3a. State-wise sales by demographic group

A heatmap shows how Kids, Women, Men and Seniors perform within each state/territory. Each labelled square contains recorded sales in millions.

In [ ]:
state_group_sales = df.groupby(['State','Group'])['Sales'].sum().unstack('Group').reindex(index=state_order,columns=customer_groups)
state_group_chart_data = state_group_sales / 1_000_000
fig,ax=plt.subplots(figsize=(10,5.5))
sns.heatmap(state_group_chart_data,annot=True,fmt='.2f',cmap='Blues',
            linewidths=.6,linecolor='white',
            cbar_kws={'label':'Recorded sales values (millions)'},ax=ax)
ax.set(title='State by demographic sales | Q4 2020',
       xlabel='Demographic group',ylabel='State / territory')
fig.tight_layout()
plt.show()

**Interpretation:** darker cells indicate larger sales totals, not necessarily better margins. The lower-selling states still show contributions from all four demographic groups.

### 3b. Group-wise sales across states

A grouped bar chart answers the reverse question: where does each demographic's recorded sales come from? The legend identifies every state.

In [ ]:
group_state_chart_data=state_group_chart_data.T
fig,ax=plt.subplots(figsize=(12,5.5))
group_state_chart_data.plot(kind='bar',ax=ax,width=.82)
ax.set(title='Sales by demographic group across states | Q4 2020',
       xlabel='Customer group',ylabel='Recorded sales values (millions)')
ax.set_xticklabels(group_state_chart_data.index,rotation=0)
ax.legend(title='State / territory',bbox_to_anchor=(1.02,1),loc='upper left')
ax.grid(axis='y',alpha=.25)
fig.tight_layout()
plt.show()

### 3c. Time-of-day analysis

The source records **Morning, Afternoon and Evening**, not exact hourly timestamps. Equal numbers of records occur in each daypart. We identify the highest and lowest recorded dayparts nationally, then compare each state's daypart mix using percentages.

In [ ]:
time_of_day_totals=df.groupby('Time')['Sales'].sum().reindex(time_order)
peak_period=time_of_day_totals.idxmax()
off_peak_period=time_of_day_totals.idxmin()
state_time_sales=df.groupby(['State','Time'])['Sales'].sum().unstack('Time').reindex(index=state_order,columns=time_order)
state_time_sales_share=100*state_time_sales.div(state_time_sales.sum(axis=1),axis=0)
fig,axes=plt.subplots(1,2,figsize=(14,5),gridspec_kw={'width_ratios':[1,1.45]})
axes[0].bar(time_order,time_of_day_totals/1_000_000,color=['#377eb8','#4daf4a','#984ea3'])
axes[0].set(title='Recorded sales by time of day',xlabel='Daypart',
            ylabel='Sales values (millions)')
for j,value in enumerate(time_of_day_totals/1_000_000):
    axes[0].text(j,value+1.2,f'{value:.1f}',ha='center')
axes[0].set_ylim(0,time_of_day_totals.max()/1_000_000*1.16)
sns.heatmap(state_time_sales_share,annot=True,fmt='.1f',cmap='Blues',
            cbar_kws={'label':'Share of state sales by daypart (%)'},ax=axes[1])
axes[1].set(title='Daypart mix by state (%)',xlabel='Time of day',ylabel='State / territory')
fig.tight_layout()
plt.show()
print('Peak recorded sales period:',peak_period,f'({time_of_day_totals.max():,.0f})')
print('Off-peak recorded sales period:',off_peak_period,f'({time_of_day_totals.min():,.0f})')

**Interpretation:** the gap between Morning and Evening is small. A targeted promotion could be piloted, but the source cannot prove causation, basket spending or incremental profit.

### 3d. Daily, weekly, monthly and quarterly sales charts

The following four charts use recorded values. The dates **31 October** and **31 December** are unrecorded, so the daily line must show NaN gaps rather than zeros. Weekly partial periods are marked.

**Daily:** one point per recorded date in Q4. Red dashed lines mark days for which we have no observations, not confirmed store closures.

In [ ]:
daily_chart_data=df.groupby('Date')['Sales'].sum().reindex(quarter_dates)
fig,ax=plt.subplots(figsize=(12,4.3))
ax.plot(daily_chart_data.index,daily_chart_data/1_000_000,
        color='#23638d',lw=1.8,marker='o',markersize=2.5)
for j,date in enumerate(missing_dates):
    ax.axvline(date,color='#c35145',ls='--',alpha=.8,
               label='Unrecorded date' if j==0 else None)
ax.set(xlim=(quarter_dates.min(),quarter_dates.max()),
       title='Daily recorded sales | Q4 2020',xlabel='Date',
       ylabel='Sales values (millions)')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(loc='upper left')
fig.tight_layout()
plt.show()

**Weekly:** totals are aggregated into Monday–Sunday weeks. A week with fewer than seven observed Q4 days is labelled so partial weeks are not mistaken for normal full weeks.

In [ ]:
weekly_chart_data=weekly_report.copy()
weekly_chart_data['Week_Start']=weekly_chart_data['Week'].apply(lambda p:p.start_time)
labels=weekly_chart_data['Week_Start'].dt.strftime('%d %b')
fig,ax=plt.subplots(figsize=(12,4.7))
bars=ax.bar(labels,weekly_chart_data['Total_Sales']/1_000_000,color='#377eb8')
for bar,days in zip(bars,weekly_chart_data['Recorded_Days']):
    if days<7:
        ax.annotate(f'{days} days',(bar.get_x()+bar.get_width()/2,bar.get_height()),
                    xytext=(0,4),textcoords='offset points',ha='center',color='#9a3931')
ax.set(title='Weekly recorded sales | Weeks beginning Monday',
       xlabel='Week starting',ylabel='Sales values (millions)')
ax.tick_params(axis='x',rotation=45)
ax.set_ylim(0,weekly_chart_data['Total_Sales'].max()/1_000_000*1.23)
fig.tight_layout()
plt.show()

**Monthly:** October, November and December have thirty recorded dates each. One Q4 dataset is insufficient to prove recurring seasonality.

In [ ]:
monthly_chart_data=monthly_report.copy()
monthly_labels=monthly_chart_data['Month'].dt.strftime('%b %Y')
fig,ax=plt.subplots(figsize=(8.5,4.3))
bars=ax.bar(monthly_labels,monthly_chart_data['Total_Sales']/1_000_000,
            color=['#4c78a8','#72b7b2','#e59c4c'])
for bar,v in zip(bars,monthly_chart_data['Total_Sales']/1_000_000):
    ax.text(bar.get_x()+bar.get_width()/2,v+2,f'{v:.1f}',ha='center')
ax.set(title='Monthly recorded sales | Q4 2020',xlabel='Month',
       ylabel='Sales values (millions)')
ax.set_ylim(0,monthly_chart_data['Total_Sales'].max()/1_000_000*1.2)
fig.tight_layout()
plt.show()

**Quarterly:** one bar represents Q4 2020; the source contains no other quarters, so there is no quarter-on-quarter trend.

In [ ]:
quarterly_chart_data=quarterly_report.copy().reset_index(drop=True)
fig,ax=plt.subplots(figsize=(6.5,4.3))
million=quarterly_chart_data['Total_Sales']/1_000_000
ax.bar(quarterly_chart_data['Quarter'].astype(str),million,color='#377eb8',width=.48)
for j,v in enumerate(million):
    ax.text(j,v+7,f'{v:.2f}M',ha='center',fontweight='bold')
ax.set(title='Q4 2020 total recorded sales | One quarter',
       xlabel='Quarter',ylabel='Sales values (millions)')
ax.set_ylim(0,million.max()*1.23)
fig.tight_layout()
plt.show()

### 3e. Descriptive statistical plots

The course also asks for a **box plot** and a Seaborn distribution. A box plot shows a median, the middle 50%, whiskers and possible outliers, using units per aggregated record.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.8))
sns.boxplot(data=df,x='Group',y='Unit',order=customer_groups,
            color='#7da6c8',ax=ax)
ax.set(title='Units per recorded row by demographic',
       xlabel='Customer group',ylabel='Units per recorded row')
fig.tight_layout()
plt.show()

Seaborn's **histplot()** displays a distribution of units per recorded row; it replaces the deprecated distplot(). These records are not individual customer baskets.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.8))
sns.histplot(data=df,x='Unit',bins=24,color='#377eb8',ax=ax)
ax.set(title='Recorded unit-volume distribution',
       xlabel='Units per recorded row',ylabel='Number of recorded rows')
fig.tight_layout()
plt.show()

### 3f. Executive visual dashboard

The notebook's plots are also combined in one four-panel dashboard for the head of Sales & Marketing: state-demographic heatmap, state totals, time of day and monthly totals.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(15,10))
sns.heatmap(state_group_chart_data,annot=True,fmt='.1f',cmap='Blues',
            linewidths=.4,cbar=False,ax=axes[0,0])
axes[0,0].set(title='By state and demographic (millions)',
              xlabel='Customer group',ylabel='State / territory')
state_summary=state_report.sort_values('Total_Sales')
axes[0,1].barh(state_summary['State'],state_summary['Total_Sales']/1_000_000,
               color='#377eb8')
axes[0,1].set(title='State/territory totals',xlabel='Sales values (millions)',
              ylabel='State / territory')
axes[1,0].bar(time_order,time_of_day_totals/1_000_000,color='#2c866f')
axes[1,0].set(title='Time of day (national)',xlabel='Daypart',
              ylabel='Sales values (millions)')
axes[1,1].bar(monthly_labels,monthly_chart_data['Total_Sales']/1_000_000,
               color='#b88645')
axes[1,1].set(title='Monthly totals',xlabel='Month',
              ylabel='Sales values (millions)')
fig.suptitle('AAL | Q4 2020 recorded sales dashboard',
             fontsize=17,fontweight='bold')
fig.tight_layout(rect=(0,0,1,.96))
plt.show()

**Why Seaborn + Matplotlib?** Seaborn is the course's preferred package and makes heatmaps, box plots and distributions easy to understand. Matplotlib provides date formatting, annotations, layout control and the dashboard. Axis titles, numeric labels and legends support accessible reading beyond colour.

**Limitations:** The source lacks profits, customer transactions, basket values, store counts and exact hourly sales. Two whole dates are missing. Recorded sales alone do not justify dropping any demographic range.

**Section 3 validation:** Chart totals must reconcile to original recorded sales. Missing daily observations must remain NaN instead of zeros.

In [ ]:
expected_total_sales=df['Sales'].sum()
assert state_group_sales.to_numpy().sum()==expected_total_sales
assert abs(group_state_chart_data.to_numpy().sum()-expected_total_sales/1_000_000)<1e-9
assert time_of_day_totals.sum()==expected_total_sales
assert (state_time_sales.sum(axis=1)==state_group_sales.sum(axis=1)).all()
assert len(daily_chart_data)==len(quarter_dates)==92
assert daily_chart_data.isna().sum()==2
assert pd.isna(daily_chart_data.loc[pd.Timestamp('2020-10-31')])
assert pd.isna(daily_chart_data.loc[pd.Timestamp('2020-12-31')])
assert daily_chart_data.sum()==expected_total_sales
for rep in (weekly_chart_data,monthly_chart_data,quarterly_chart_data):
    assert rep['Total_Sales'].sum()==expected_total_sales
print('Section 3 checks passed: chart totals reconcile and missing dates remain gaps.')

---
## 4. Report Generation

**To be completed last.** We will bring together methods, data tables, plots, limitations and evidence-based sales recommendations. The current source does not justify eliminating demographic groups solely based on revenue.

### Lead candidate recommendation for Section 4 (approved for inclusion)

**Pilot a targeted November campaign, potentially around Black Friday, and assess incremental profit before expanding it.** The supplied data records November sales of **90,682,500**, compared with **114,290,000 in October** and **135,330,000 in December**. These observations motivate a test, not a claim that discounting will succeed.

Potential initial markets to investigate are **Women's merchandise in WA** and **Seniors' merchandise in NT**, with comparison/control groups and careful stock planning. Evaluate margin after discounts, campaign cost, returns, incremental units, and whether some demand merely shifts from December. Do **not** recommend withdrawing demographic categories on recorded sales values alone.

**Caveat:** customer transactions, prices, gross margin and past promotions are unavailable; a single quarter cannot establish recurring seasonality. The full experimental design and supporting caveats are in `04_november_promotion_proposal.md`. The remainder of Section 4 is still to be written.